In [6]:
import pandas as pd

data_path = r"D:\AQI\AQI_Project\Data\air_quality.xlsx"
df = pd.read_excel(data_path)

df.head()


,Year,Day of the Year,SO2 µg/m3,Nox µg/m3,RSPM µg/m3,precipMM,maxtempC,mintempC,sunHour,uvIndex,WindGustKmph,humidity,pressure,tempC,windspeedKmph,AQI
0,2009,1,34,69,213,0.0,31,17,11.0,6,11.083333,37.875000,1012.791667,22.541667,4.500000,175
1,2009,2,18,66,228,0.0,30,17,11.0,6,11.833333,44.000000,1013.750000,22.125000,5.458333,185
2,2009,3,15,42,171,0.0,30,17,11.0,6,9.375000,47.791667,1014.708333,22.583333,3.750000,147
3,2009,9,10,39,206,0.0,28,19,11.0,6,10.208333,54.625000,1015.625000,22.875000,5.208333,171
4,2009,11,24,45,138,0.0,30,20,11.0,6,10.208333,44.666667,1015.000000,24.541667,4.333333,125


In [3]:
df.isnull().sum()

Year               0
Day of the Year    0
SO2 µg/m3          0
Nox µg/m3          0
RSPM µg/m3         0
precipMM           0
maxtempC           0
mintempC           0
sunHour            0
uvIndex            0
WindGustKmph       0
humidity           0
pressure           0
tempC              0
windspeedKmph      0
AQI                0
dtype: int64

In [7]:
df = df.dropna()
df.shape

(2249, 16)

In [8]:
# Create average temperature
df["temp_avg"] = (df["maxtempC"] + df["mintempC"]) / 2

# Create lag features for time series
df["AQI_lag1"] = df["AQI"].shift(1)
df["AQI_lag7"] = df["AQI"].shift(7)

# Drop NA created by lagging
df = df.dropna()

df.head()

,Year,Day of the Year,SO2 µg/m3,Nox µg/m3,RSPM µg/m3,precipMM,maxtempC,mintempC,sunHour,uvIndex,WindGustKmph,humidity,pressure,tempC,windspeedKmph,AQI,temp_avg,AQI_lag1,AQI_lag7
7,2009,14,24,54,118,0.0,31,19,11.0,6,15.500000,37.125000,1018.416667,23.666667,8.291667,112,25.0,131.0,175.0
8,2009,15,15,69,220,0.0,30,19,11.0,6,17.916667,36.791667,1017.541667,23.500000,8.708333,180,24.5,112.0,185.0
9,2009,16,27,49,129,0.0,30,19,11.0,6,14.541667,38.166667,1017.083333,23.500000,7.958333,119,24.5,180.0,147.0
10,2009,18,36,70,173,0.0,31,18,11.0,6,12.166667,40.250000,1014.375000,23.750000,6.125000,149,24.5,119.0,171.0
11,2009,19,31,41,158,0.0,31,19,11.0,6,13.708333,35.500000,1014.208333,24.166667,6.875000,139,25.0,149.0,125.0


In [9]:
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split

X = df[["temp_avg", "humidity", "windspeedKmph", "AQI_lag1", "AQI_lag7"]]
y = df["AQI"]

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.2)

model = XGBRegressor(n_estimators=100, learning_rate=0.1)
model.fit(X_train, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [10]:
from sklearn.metrics import mean_absolute_error

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)

print("MAE:", mae)


MAE: 17.674217224121094


In [11]:
import os

os.makedirs("../models", exist_ok=True)
print("models folder is ready")

models folder is ready


In [12]:
import joblib

joblib.dump(model, "../models/best_model.pkl")
print("Model saved successfully in models folder")

Model saved successfully in models folder


In [13]:
import os

os.listdir("../models")

['best_model.pkl']

In [14]:
import joblib
import pandas as pd
from flask import Flask, request, jsonify

app = Flask(__name__)

# Load trained model
model = joblib.dump(model, r"D:/AQI/AQI_Project/models/best_model.pkl")

@app.route("/predict", methods=["POST"])
def predict():
    data = request.json
    
    df = pd.DataFrame([data])
    
    prediction = model.predict(df)
    
    return jsonify({
        "Predicted_AQI": float(prediction[0])
    })

if __name__ == "__main__":
    app.run(debug=True)


 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
 * Restarting with watchdog (windowsapi)


SystemExit: 1

C:\Users\Sushant\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
